# 04 EDA Open Source

Generates source coverage, temporal coverage, missingness, and numeric correlation summaries for public features only.

In [15]:
import importlib.util  # Import modules directly from file paths.
import sys  # Register imported modules for cross-module imports.
from pathlib import Path  # Work with filesystem paths.

PROJECT_ROOT = Path("/content/HDX-sources-and-more-API-connection")  # Set repo root.
SRC_DIR = PROJECT_ROOT / "src"  # Set src folder.

def load_module(module_name, module_path):  # Load one Python file as a module.
    spec = importlib.util.spec_from_file_location(module_name, module_path)  # Create import spec.
    module = importlib.util.module_from_spec(spec)  # Create module object.
    sys.modules[module_name] = module  # Register module so other files can import it.
    spec.loader.exec_module(module)  # Execute module code.
    return module  # Return loaded module.

paths = load_module("paths", SRC_DIR / "paths.py")  # Load paths.py first.
cleaning = load_module("cleaning", SRC_DIR / "cleaning.py")  # Load cleaning.py second.
feature_assembly = load_module("feature_assembly", SRC_DIR / "feature_assembly.py")  # Load feature assembly third.

CLEAN_DIR = paths.CLEAN_DIR  # Get cleaned output folder.
MODEL_FEATURES_DIR = paths.MODEL_FEATURES_DIR  # Get model feature folder.
BASE_FEATURE_TABLE_PATH = MODEL_FEATURES_DIR / "base_feature_table_country_month_year.csv"  # Set base table path.

print("Direct module imports worked.")  # Confirm imports.
print(f"Cleaned CSV count before 03c: {len(list(CLEAN_DIR.glob('*.csv')))}")  # Show cleaned file count.

if not list(CLEAN_DIR.glob("*.csv")):  # Regenerate cleaned files if missing.
    summary_report, reject_rows = cleaning.clean_silver_directory()  # Run 03c cleaning.

assembled_features, assembly_report, feature_catalog = feature_assembly.assemble_feature_table()  # Run 03d assembly.

print(f"Base feature table exists: {BASE_FEATURE_TABLE_PATH.exists()}")  # Confirm output exists.
print(f"Rows: {len(assembled_features):,}")  # Show row count.
print(f"Columns: {len(assembled_features.columns):,}")  # Show column count.

Direct module imports worked.
Cleaned CSV count before 03c: 0
Base sources merged: 7
Wide sources deferred: ['clean_whs2026_country_month.csv', 'clean_worldriskindex_country_month.csv']
Base feature rows: 46,522
Base feature columns: 24
Wrote: /content/HDX-sources-and-more-API-connection/outputs/model_features/base_feature_table_country_month_year.csv
Base feature table exists: True
Rows: 46,522
Columns: 24


In [16]:
import pandas as pd  # Load pandas for EDA.

base_features = pd.read_csv(BASE_FEATURE_TABLE_PATH, low_memory=False)  # Read base feature table.

print(f"Rows: {len(base_features):,}")  # Show row count.
print(f"Columns: {len(base_features.columns):,}")  # Show column count.
print(f"Duplicate country-month rows: {base_features.duplicated(['iso3', 'country', 'year', 'month']).sum()}")  # Check key uniqueness.

base_features.head()  # Preview table.

Rows: 46,522
Columns: 24
Duplicate country-month rows: 0


,iso3,country,year,month,civilian_targeting_events,civilian_targeting_fatalities,demonstration_events_events,gdacs_event_count,gdacs_max_severity_value,gdacs_mean_severity_value,...,inform_risk_annual_carried_monthly,political_violence_events,political_violence_fatalities,views_conflict_forecasts_views_main_mean,views_conflict_forecasts_views_main_dich,views_conflict_forecasts_views_main_mean_ln,who_covid_covid_new_cases,who_covid_covid_new_deaths,who_covid_covid_cumulative_cases,who_covid_covid_cumulative_deaths
0,AD,Andorra,2020,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
1,AD,Andorra,2020,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2,AD,Andorra,2020,3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,376.0,12.0,376.0,12.0
3,AD,Andorra,2020,4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,368.0,30.0,744.0,42.0
4,AD,Andorra,2020,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,20.0,9.0,764.0,51.0


In [17]:
missingness = (  # Build missingness summary.
    base_features
    .isna()
    .mean()
    .reset_index(name="missing_rate")
    .rename(columns={"index": "column"})
    .sort_values("missing_rate", ascending=False)
)

missingness.head(25)  # Show most-missing columns.

,column,missing_rate
7,gdacs_event_count,0.998753
8,gdacs_max_severity_value,0.998753
9,gdacs_mean_severity_value,0.998753
18,views_conflict_forecasts_views_main_dich,0.978505
17,views_conflict_forecasts_views_main_mean,0.978505
19,views_conflict_forecasts_views_main_mean_ln,0.978505
4,civilian_targeting_events,0.869739
6,demonstration_events_events,0.869739
5,civilian_targeting_fatalities,0.869739
16,political_violence_fatalities,0.869739


In [18]:
coverage = base_features.groupby("year").agg(  # Summarize yearly coverage.
    country_count=("iso3", "nunique"),
    row_count=("iso3", "size"),
).reset_index()

coverage  # Display year-level coverage.

,year,country_count,row_count
0,1997,13,156
1,1998,13,156
2,1999,13,156
3,2000,13,156
4,2001,13,156
5,2002,13,156
6,2003,13,156
7,2004,13,156
8,2005,13,156
9,2006,13,156


In [19]:
coverage.head(25)

,year,country_count,row_count
0,1997,13,156
1,1998,13,156
2,1999,13,156
3,2000,13,156
4,2001,13,156
5,2002,13,156
6,2003,13,156
7,2004,13,156
8,2005,13,156
9,2006,13,156


In [21]:
numeric_features = base_features.drop(
    columns=["year", "month"],
    errors="ignore"
).select_dtypes(include="number")

correlations = numeric_features.corr().stack().reset_index()
correlations.columns = ["feature_1", "feature_2", "correlation"]
correlations = correlations[correlations["feature_1"] < correlations["feature_2"]]

correlations.sort_values(
    "correlation",
    key=lambda s: s.abs(),
    ascending=False
).head(25)

,feature_1,feature_2,correlation
79,gdacs_mean_severity_value,inform_risk_inform_inform,-1.000000
64,gdacs_max_severity_value,inform_risk_inform_inform,-1.000000
63,gdacs_max_severity_value,inform_risk_inform_ha,1.000000
65,gdacs_max_severity_value,inform_risk_inform_vu,-1.000000
80,gdacs_mean_severity_value,inform_risk_inform_vu,-1.000000
78,gdacs_mean_severity_value,inform_risk_inform_ha,1.000000
62,gdacs_max_severity_value,inform_risk_inform_cc,-1.000000
77,gdacs_mean_severity_value,inform_risk_inform_cc,-1.000000
61,gdacs_max_severity_value,gdacs_mean_severity_value,0.994506
142,political_violence_events,views_conflict_forecasts_views_main_mean,0.954275


In [22]:
correlations.sort_values('correlation', key=lambda s: s.abs(), ascending=False).head(25) if not correlations.empty else correlations

,feature_1,feature_2,correlation
79,gdacs_mean_severity_value,inform_risk_inform_inform,-1.000000
64,gdacs_max_severity_value,inform_risk_inform_inform,-1.000000
63,gdacs_max_severity_value,inform_risk_inform_ha,1.000000
65,gdacs_max_severity_value,inform_risk_inform_vu,-1.000000
80,gdacs_mean_severity_value,inform_risk_inform_vu,-1.000000
78,gdacs_mean_severity_value,inform_risk_inform_ha,1.000000
62,gdacs_max_severity_value,inform_risk_inform_cc,-1.000000
77,gdacs_mean_severity_value,inform_risk_inform_cc,-1.000000
61,gdacs_max_severity_value,gdacs_mean_severity_value,0.994506
142,political_violence_events,views_conflict_forecasts_views_main_mean,0.954275


| Pattern                                                                             | Interpretation                                                                                   |
| ----------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------ |
| `gdacs_max_severity_value` vs `gdacs_mean_severity_value` = `0.9945`                | Expected. Max and mean severity are nearly the same because GDACS has sparse monthly events.     |
| GDACS vs INFORM = `1.0` / `-1.0`                                                    | Suspicious. Likely caused by very few overlapping non-null rows, not a true global relationship. |
| `political_violence_events` vs `views_conflict_forecasts_views_main_mean` = `0.954` | Plausible and important. VIEWS forecasts are strongly related to observed conflict intensity.    |
| `views_main_dich` vs `views_main_mean_ln` = `0.922`                                 | Expected. They are alternate encodings of the same source signal.                                |
| COVID cumulative cases vs deaths = `0.781`                                          | Expected. Cumulative pandemic measures move together.                                            |


In [23]:
correlation_rows = []  # Store pairwise correlation diagnostics.

for feature_1 in numeric_features.columns:  # Loop over first feature.
    for feature_2 in numeric_features.columns:  # Loop over second feature.
        if feature_1 >= feature_2:  # Keep each pair only once.
            continue

        pair_data = numeric_features[[feature_1, feature_2]].dropna()  # Keep rows where both features exist.

        if len(pair_data) < 2:  # Skip pairs that cannot support correlation.
            continue

        correlation_rows.append({
            "feature_1": feature_1,
            "feature_2": feature_2,
            "overlap_rows": len(pair_data),
            "correlation": pair_data[feature_1].corr(pair_data[feature_2]),
        })  # Save diagnostic row.

correlation_diagnostics = pd.DataFrame(correlation_rows)  # Build diagnostics table.

correlation_diagnostics.sort_values(
    ["correlation", "overlap_rows"],
    key=lambda s: s.abs() if s.name == "correlation" else s,
    ascending=False
).head(25)

/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:299

,feature_1,feature_2,overlap_rows,correlation
51,gdacs_max_severity_value,inform_risk_inform_cc,2,-1.000000
52,gdacs_max_severity_value,inform_risk_inform_ha,2,1.000000
54,gdacs_max_severity_value,inform_risk_inform_vu,2,-1.000000
60,gdacs_mean_severity_value,inform_risk_inform_cc,2,-1.000000
61,gdacs_mean_severity_value,inform_risk_inform_ha,2,1.000000
63,gdacs_mean_severity_value,inform_risk_inform_vu,2,-1.000000
53,gdacs_max_severity_value,inform_risk_inform_inform,2,-1.000000
62,gdacs_mean_severity_value,inform_risk_inform_inform,2,-1.000000
50,gdacs_max_severity_value,gdacs_mean_severity_value,58,0.994506
84,political_violence_events,views_conflict_forecasts_views_main_mean,44,0.954275


In [24]:
correlation_diagnostics[
    correlation_diagnostics["overlap_rows"] >= 100
].sort_values(
    "correlation",
    key=lambda s: s.abs(),
    ascending=False
).head(25)

,feature_1,feature_2,overlap_rows,correlation
92,views_conflict_forecasts_views_main_dich,views_conflict_forecasts_views_main_mean_ln,1000,0.921650
78,inform_risk_inform_inform,inform_risk_inform_vu,22920,0.918290
70,inform_risk_inform_cc,inform_risk_inform_inform,22920,0.839431
74,inform_risk_inform_ha,inform_risk_inform_inform,22920,0.827662
71,inform_risk_inform_cc,inform_risk_inform_vu,22920,0.782550
96,who_covid_covid_cumulative_cases,who_covid_covid_cumulative_deaths,18881,0.780542
83,political_violence_events,political_violence_fatalities,6060,0.776926
9,civilian_targeting_events,political_violence_events,6060,0.713998
10,civilian_targeting_events,political_violence_fatalities,6060,0.656014
0,civilian_targeting_events,civilian_targeting_fatalities,6060,0.641439


| Finding                                             | Meaning                                                     |
| --------------------------------------------------- | ----------------------------------------------------------- |
| INFORM subcomponents are highly correlated          | Expected. Same source family, overlapping risk dimensions.  |
| VIEWS transformed fields are correlated             | Expected. Same forecast signal represented different ways.  |
| COVID cumulative cases/deaths correlate             | Expected. Same source family.                               |
| Political violence events/fatalities correlate      | Useful signal, but probably redundant together.             |
| Civilian targeting and political violence correlate | Plausible conflict-intensity relationship.                  |
| GDACS should be handled carefully                   | Too sparse for simple correlation interpretation right now. |


In [25]:
meaningful_correlations = correlation_diagnostics[
    correlation_diagnostics["overlap_rows"] >= 100
].sort_values(
    "correlation",
    key=lambda s: s.abs(),
    ascending=False
)  # Keep only correlations with enough overlapping rows.

meaningful_correlations.head(25)  # Preview strongest meaningful correlations.

,feature_1,feature_2,overlap_rows,correlation
92,views_conflict_forecasts_views_main_dich,views_conflict_forecasts_views_main_mean_ln,1000,0.921650
78,inform_risk_inform_inform,inform_risk_inform_vu,22920,0.918290
70,inform_risk_inform_cc,inform_risk_inform_inform,22920,0.839431
74,inform_risk_inform_ha,inform_risk_inform_inform,22920,0.827662
71,inform_risk_inform_cc,inform_risk_inform_vu,22920,0.782550
96,who_covid_covid_cumulative_cases,who_covid_covid_cumulative_deaths,18881,0.780542
83,political_violence_events,political_violence_fatalities,6060,0.776926
9,civilian_targeting_events,political_violence_events,6060,0.713998
10,civilian_targeting_events,political_violence_fatalities,6060,0.656014
0,civilian_targeting_events,civilian_targeting_fatalities,6060,0.641439


Initial public-source EDA confirms the base Gold feature table is valid at country-month grain with no duplicate keys. Correlation review shows expected redundancy within source families such as INFORM, VIEWS, COVID, and conflict-event features. Sparse sources such as GDACS require caution because extreme correlations can be driven by very small overlap counts rather than true relationships.

| Signal group             | What it tells us                                                                                                                                          |
| ------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------- |
| VIEWS fields             | Several VIEWS forecast columns are alternate versions of the same forecast signal, so we may later keep only one or let regularization handle redundancy. |
| INFORM fields            | INFORM risk dimensions are highly related, which is expected because they come from the same index system.                                                |
| COVID cumulative fields  | Cumulative cases and deaths move together, so they may be redundant unless COVID is retained as context.                                                  |
| Conflict event fields    | Political violence, fatalities, and civilian targeting are meaningfully correlated and likely form the strongest public-source crisis-intensity signal.   |
| INFORM + conflict fields | Some moderate relationships exist, suggesting structural risk may align with observed violence, but not perfectly.                                        |


In [27]:
country_coverage = (
    base_features
    .groupby(["iso3", "country"], dropna=False)
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        first_month=("month", "min"),
        last_month=("month", "max"),
        country_month_rows=("month", "size"),
    )
    .reset_index()
    .sort_values(["iso3", "country"])
)  # Summarize coverage by country.

country_coverage.head(25)  # Preview country coverage.

,iso3,country,first_year,last_year,first_month,last_month,country_month_rows
0,AD,Andorra,2020,2026,1,12,79
1,AE,United Arab Emirates,2020,2026,1,12,79
2,AF,Afghanistan,2020,2026,1,12,79
3,AFG,Afghanistan,2016,2026,1,12,131
4,AG,Antigua and Barbuda,2020,2026,1,12,79
5,AGO,Angola,2016,2026,1,12,125
6,AI,Anguilla,2020,2026,1,12,79
7,AL,Albania,2020,2026,1,12,79
8,ALB,Albania,2016,2026,1,12,125
9,ALB,"Albania, Austria, Bosnia & Herzegovina, Belgiu...",2025,2025,12,12,1


In [30]:
numeric_summary = base_features.select_dtypes(include="number").describe().T  # Summarize numeric features.

numeric_summary.head(25)  # Preview numeric summary.

,count,mean,std,min,25%,50%,75%,max
year,46522.0,2.020632e+03,4.900793e+00,1997.0000,2019.000000,2022.00000,2024.000000,2.026000e+03
month,46522.0,6.458020e+00,3.434389e+00,1.0000,3.000000,6.00000,9.000000,1.200000e+01
civilian_targeting_events,6060.0,4.418515e+01,9.011148e+01,0.0000,1.000000,11.00000,46.000000,1.078000e+03
civilian_targeting_fatalities,6060.0,7.801518e+01,2.411928e+02,0.0000,0.000000,14.00000,71.000000,7.376000e+03
demonstration_events_events,6060.0,2.518086e+01,7.849304e+01,0.0000,0.000000,3.00000,16.000000,1.769000e+03
gdacs_event_count,58.0,8.206897e+00,2.249971e+01,1.0000,1.000000,1.00000,2.000000,1.280000e+02
gdacs_max_severity_value,58.0,1.412815e+05,3.162626e+05,0.0000,5.800000,10598.50000,65608.750000,1.487749e+06
gdacs_mean_severity_value,58.0,1.298341e+05,3.186504e+05,0.0000,3.887500,7800.50000,15718.937500,1.487749e+06
inform_risk_inform_cc,22920.0,4.486859e+00,1.908793e+00,0.7000,3.100000,4.30000,6.000000,9.500000e+00
inform_risk_inform_ha,22920.0,3.841309e+00,2.164439e+00,0.5000,2.100000,3.30000,5.300000,8.700000e+00


In [31]:
EDA_DIR = OUTPUTS_DIR / "eda_reports"  # Set EDA output folder.
EDA_DIR.mkdir(parents=True, exist_ok=True)  # Create EDA output folder if needed.

missingness.to_csv(EDA_DIR / "base_feature_missingness.csv", index=False)  # Save missingness report.
coverage.to_csv(EDA_DIR / "base_feature_year_coverage.csv", index=False)  # Save year coverage report.
country_coverage.to_csv(EDA_DIR / "base_feature_country_coverage.csv", index=False)  # Save country coverage report.
numeric_summary.to_csv(EDA_DIR / "base_feature_numeric_summary.csv")  # Save numeric summary report.
meaningful_correlations.to_csv(EDA_DIR / "base_feature_meaningful_correlations.csv", index=False)  # Save filtered correlations.

print(f"EDA reports written to: {EDA_DIR}")  # Show output folder.
print(list(EDA_DIR.glob("*.csv")))  # Confirm files written.

EDA reports written to: /content/HDX-sources-and-more-API-connection/outputs/eda_reports
[PosixPath('/content/HDX-sources-and-more-API-connection/outputs/eda_reports/base_feature_year_coverage.csv'), PosixPath('/content/HDX-sources-and-more-API-connection/outputs/eda_reports/base_feature_country_coverage.csv'), PosixPath('/content/HDX-sources-and-more-API-connection/outputs/eda_reports/base_feature_meaningful_correlations.csv'), PosixPath('/content/HDX-sources-and-more-API-connection/outputs/eda_reports/base_feature_missingness.csv'), PosixPath('/content/HDX-sources-and-more-API-connection/outputs/eda_reports/base_feature_numeric_summary.csv')]


Public-source EDA identified expected redundancy within VIEWS, INFORM, COVID, and conflict-event feature families. The meaningful correlation table now filters out sparse overlaps, so extreme GDACS correlations are excluded from interpretation. This supports the next step: feature selection/engineering before target-joined modeling in Databricks.